In [25]:
import pandas as pd
import numpy as np

In [26]:
# Load the data
raw = pd.read_csv('gcdg_zaf.csv', low_memory=False)
print(raw.shape)
print(raw['subjid'].nunique(), 'children')

(4176, 363)
2389 children


In [27]:
print(raw['agedays'].describe())
print('missing age:', raw['agedays'].isna().sum())

count    4175.000000
mean      943.178683
std       450.780117
min       125.000000
25%       721.000000
50%       782.000000
75%      1461.000000
max      1461.000000
Name: agedays, dtype: float64
missing age: 1


In [28]:
df = raw[raw['agedays'] <= 3 * 365].copy()
df['age_months'] = df['agedays'] / 30.4375
print('rows:', len(df))
print('children:', df['subjid'].nunique())
print(df['age_months'].min(), df['age_months'].max())

rows: 2561
children: 2033
4.1067761806981515 29.305954825462013


In [29]:
# Checking Data Quality
meta_cols = ['ctrycd', 'cohort', 'cohortn', 'subjid', 'agedays', 'sex', 'gagebrth']
item_cols = [col for col in df.columns if col not in meta_cols + ['age_months']]
print(len(item_cols), 'item columns')
print(df[meta_cols].isna().sum())

356 item columns
ctrycd       0
cohort       0
cohortn      0
subjid       0
agedays      0
sex          0
gagebrth    53
dtype: int64


In [30]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate child + age:', df.duplicated(['subjid', 'agedays']).sum())

duplicate rows: 0
duplicate child + age: 0


In [31]:
values = pd.unique(df[item_cols].values.ravel())
print(sorted(v for v in values if v == v))

[np.float64(0.0), np.float64(1.0)]


In [32]:
print(df['sex'].value_counts())
print('children with more than one sex:', (df.groupby('subjid')['sex'].nunique() > 1).sum())
print('children with more than one gestational age:', (df.groupby('subjid')['gagebrth'].nunique() > 1).sum())

sex
Female    1312
Male      1249
Name: count, dtype: int64
children with more than one sex: 0
children with more than one gestational age: 0


In [33]:
answered = df[item_cols].notna().sum(axis=1)
print(answered.describe())
print('rows with no items answered:', (answered == 0).sum())

count    2561.000000
mean       72.866458
std        69.354925
min         8.000000
25%        28.000000
50%        29.000000
75%       166.000000
max       201.000000
dtype: float64
rows with no items answered: 0


In [34]:
never_answered = [col for col in item_cols if df[col].isna().all()]
print(len(never_answered), 'items never answered:', never_answered)

21 items never answered: ['vinxxc031', 'vinxxc032', 'vinxxc033', 'vinxxc034', 'vinxxc035', 'vinxxc036', 'vinxxc037', 'vinxxc038', 'vinxxc039', 'vinxxc040', 'vinxxc041', 'vinxxc042', 'vinxxc043', 'vinxxc044', 'vinxxc045', 'vinxxc046', 'vinxxc047', 'vinxxc048', 'vinxxc049', 'vinxxc050', 'vinxxc051']


In [35]:
vineland = [col for col in item_cols if col.startswith('vin')]
others = [col for col in item_cols if not col.startswith('vin')]
has_vineland = df[vineland].notna().any(axis=1)
has_others = df[others].notna().any(axis=1)
print('Vineland only:', (has_vineland & ~has_others).sum())
print('other instruments only:', (~has_vineland & has_others).sum())
print('both:', (has_vineland & has_others).sum())

Vineland only: 1801
other instruments only: 760
both: 0


In [36]:
print(df.groupby('subjid').size().value_counts())
print(pd.cut(df['age_months'], [0, 6, 12, 18, 24, 30]).value_counts().sort_index())

1    1551
2     436
3      46
Name: count, dtype: int64
age_months
(0, 6]       216
(6, 12]      274
(12, 18]     270
(18, 24]     526
(24, 30]    1275
Name: count, dtype: int64


In [37]:
item_cols = [col for col in item_cols if col not in never_answered]
print(len(item_cols), 'items kept')

335 items kept


In [38]:
import json

labels = json.load(open('item_labels.json'))
print(len(labels), 'labels')
print('items without a label:', [col for col in item_cols if col not in labels])

335 labels
items without a label: []


In [39]:
prefix_domain = {
    'by1pdd': 'motor',
    'by1mdd': 'cognitive',
    'gricgd': 'cognitive',
    'griehd': 'motor',
    'grigmd': 'motor',
    'grihsd': 'language',
    'sgrehd': 'motor',
    'sgrfmd': 'motor',
    'sgrgmd': 'motor',
    'sgrhsd': 'language',
    'sgrred': 'social_emotional',
    'vinxxc': 'social_emotional',
}

In [40]:
overrides = {
    'by1mdd': {
        'motor': [37, 44, 46, 49, 51, 54, 56, 60, 64, 69, 70, 77, 82, 87, 98, 100, 103, 105, 107, 108, 109, 111, 112, 114, 118, 119, 123, 125],
        'language': [47, 48, 55, 79, 84, 85, 89, 90, 94, 101, 106, 113, 116, 124],
        'social_emotional': [42, 53, 58, 61, 65, 76, 81, 97, 117],
    },
    'gricgd': {
        'motor': [4, 7, 8, 11, 12, 13, 23],
    },
    'griehd': {
        'cognitive': [5, 6, 7, 8, 13, 22, 208, 210],
    },
    'sgrehd': {
        'cognitive': [20, 25],
    },
    'sgrfmd': {
        'cognitive': [8],
    },
    'grihsd': {
        'cognitive': [201, 210],
    },
    'sgrhsd': {
        'social_emotional': [10],
        'cognitive': [15],
    },
    'sgrred': {
        'language': [6, 13, 30],
        'cognitive': [18, 29, 32, 37],
    },
    'vinxxc': {
        'motor': [4, 8, 11, 14, 15, 25],
        'cognitive': [5, 12, 22],
        'language': [6, 13, 16, 24],
    },
}

In [41]:
item_map = {}
for item in item_cols:
    item_map[item] = prefix_domain[item[:6]]

for prefix, groups in overrides.items():
    for domain, numbers in groups.items():
        for number in numbers:
            item_map[prefix + str(number).zfill(3)] = domain

map_table = pd.DataFrame({'item': item_cols})
map_table['prefix'] = map_table['item'].str[:6]
map_table['domain'] = map_table['item'].map(item_map)
map_table['label'] = map_table['item'].map(labels)
print(map_table['domain'].value_counts())

domain
motor               146
cognitive            82
social_emotional     54
language             53
Name: count, dtype: int64


In [42]:
print(pd.crosstab(map_table['prefix'], map_table['domain']))

domain  cognitive  language  motor  social_emotional
prefix                                              
by1mdd         37        14     28                 9
by1pdd          0         0     39                 0
gricgd         24         0      7                 0
griehd          8         0     22                 0
grigmd          0         0     30                 0
grihsd          2        29      0                 0
sgrehd          2         0      4                 0
sgrfmd          1         0      4                 0
sgrgmd          0         0      6                 0
sgrhsd          1         3      0                 1
sgrred          4         3      0                28
vinxxc          3         4      6                16


In [43]:
for domain in ['social_emotional', 'language', 'cognitive', 'motor']:
    print('\n' + domain.upper())
    subset = map_table[map_table['domain'] == domain]
    for _, row in subset.iterrows():
        print(row['item'], '-', row['label'])


SOCIAL_EMOTIONAL
by1mdd042 - Aware of strange situation
by1mdd053 - mirror image approach
by1mdd058 - Discriminates strangers
by1mdd061 - Likes frolic play
by1mdd065 - smiles at mirror (5.4 months)
by1mdd076 - playful response to mirror
by1mdd081 - cooperates in games (is this some other scale?)
by1mdd097 - repeats performance laughed at
by1mdd117 - shows shoes or other clothing, or own toy
sgrhsd010 - Laughs aloud
sgrred007 - Follows moving person
sgrred008 - Coos and smiles
sgrred009 - Friendly to strangers
sgrred010 - Resist ring being taken away
sgrred011 - Frolics when played with
sgrred012 - Stops crying when spoken to
sgrred014 - Anticipate being lifted
sgrred015 - Holds a spoon
sgrred016 - Stretches to be taken
sgrred017 - Drinks from a cup
sgrred019 - Reacts to mirror image
sgrred020 - Knows strangers
sgrred021 - Enthusiastic
sgrred022 - Displeased when toy taken
sgrred023 - Helps hold cup
sgrred024 - Pulls off hat
sgrred025 - Smiles, plays mirror
sgrred026 - Waves bye bye
sg

In [44]:
domains = ['cognitive', 'motor', 'language', 'social_emotional']
for domain in domains:
    cols = map_table.loc[map_table['domain'] == domain, 'item']
    df['n_' + domain] = df[cols].notna().sum(axis=1)

df['family'] = np.where(has_vineland, 'vineland', 'other')
print(df.groupby('family')[['n_' + d for d in domains]].describe().T.loc[(slice(None), ['min', 'mean', 'max']), :])

family                       other   vineland
n_cognitive        min   39.000000   1.000000
n_motor            min   80.000000   1.000000
n_language         min   24.000000   0.000000
n_social_emotional min   22.000000   4.000000
n_cognitive        mean  43.700000   2.951694
n_motor            mean  83.980263   5.542476
n_language         mean  28.339474   3.980566
n_social_emotional mean  22.638158  15.749028
n_cognitive        max   52.000000   3.000000
n_motor            max   91.000000   6.000000
n_language         max   36.000000   4.000000
n_social_emotional max   23.000000  16.000000


In [45]:
json.dump(item_map, open('item_domain_map.json', 'w'), indent=1)

In [46]:
min_items = 3

for domain in domains:
    cols = map_table.loc[map_table['domain'] == domain, 'item']
    df[domain] = df[cols].mean(axis=1)
    df.loc[df['n_' + domain] < min_items, domain] = np.nan

print(df[domains].describe())

         cognitive        motor     language  social_emotional
count  2481.000000  2560.000000  2554.000000       2561.000000
mean      0.838983     0.837390     0.861393          0.848855
std       0.197167     0.171259     0.179964          0.118319
min       0.000000     0.000000     0.250000          0.125000
25%       0.666667     0.744815     0.750000          0.800000
50%       0.923077     0.835165     1.000000          0.875000
75%       1.000000     1.000000     1.000000          0.937500
max       1.000000     1.000000     1.000000          1.000000


In [47]:
print(df[domains].isna().sum())
print('rows with all four domains:', df[domains].notna().all(axis=1).sum())

cognitive           80
motor                1
language             7
social_emotional     0
dtype: int64
rows with all four domains: 2481


In [48]:
print(df.groupby('family')[domains].mean())
print((df[domains] == 1).mean())

          cognitive     motor  language  social_emotional
family                                                   
other      0.801120  0.803949  0.771818          0.864424
vineland   0.855704  0.851509  0.899340          0.842286
cognitive           0.443186
motor               0.359625
language            0.508395
social_emotional    0.117142
dtype: float64


**Decisions**

- Each domain score is the proportion of answered items passed. Missing items are ignored.
- A domain score is only kept when at least 3 items were answered. Below that it is set to missing.
- With this rule, 80 Vineland rows lose the cognitive score, 7 lose language and 1 loses motor. Every other row keeps all four domains.
- Many scores equal 1.0 (about half for cognitive and language). This is a ceiling effect and matters for the norms and z-scores (issues 5 and 6).